# Baseline Pipeline

Computes all 5 corpus-wide baselines (RareCodonAnalysis, CodonAnalysis, CodonPairBiasAnalysis, GCAnalysis, KmerAnalysis) over `RefGenes/NHGeneBodySupp/` and writes `Standards/<Name>Analysis.json` -- the exact files `standards_io.load_standards()` expects.

**Supersedes** `Rare_Codons.ipynb`, `Codon_Usage.ipynb`, `Codon_Pair_Bias.ipynb`, `GC_Analysis.ipynb`, `Kmer_Analysis.ipynb`. Those notebooks each independently reread every gene JSON from disk (`Kmer_Analysis.ipynb` alone reread the whole corpus 18 times, and its real run logged over 10 hours for k=9 alone). This notebook is a thin driver -- all the actual logic lives in `src/genewriter/baseline_pipeline.py` and the `baseline_<test>.py`/`gpu_<test>_count.py` modules it dispatches to, so it's importable/testable outside a notebook too (see `tests/test_baseline_pipeline*.py`).

Per chunk of genes: loads once -- concurrently, via a thread pool (`LOAD_WORKERS` below), since the real cost of a Drive-mounted read is network/FUSE round-trip *latency* per file, not local CPU work, so threads help here despite Colab's single CPU core -- then all 5 tests run SEQUENTIALLY inside ONE shared forked process (wave 2 -- wave 1 is empty by default now, kept only in case a future test is genuinely CPU-only). Every test is GPU-capable as of 2026-08-19 (rare_codon/codon_usage/codon_pair_bias/gc via `gpu_<name>_count.py`, matching kmer's own `gpu_kmer_count.py`), so they deliberately run one after another sharing a single CUDA context rather than as separate concurrent processes -- N concurrent processes would mean N CUDA contexts fighting over one GPU's VRAM, more contention risk than real gain. One test's failure doesn't stop its siblings in the same group (each is individually try/excepted inside the shared child). Resumable at chunk granularity, both for a whole chunk and per-test within a chunk. See `baseline_pipeline.py`'s module docstring for the full design, including the fork/CUDA-safety invariant this relies on.

**Requires the POSIX `fork` multiprocessing start method** (Colab and WSL both have it; native Windows Python does not -- `run_pipeline()` raises a clear error rather than silently falling back to something slower/unsafe).

In [ ]:
# Clones this repo fresh into the Colab VM's local disk (not Drive) -- the
# same proven pattern Testing_GeneWriter.ipynb uses to make `genewriter`
# importable. Relying on `src/genewriter` already being present via some
# external Drive-sync mechanism (what an earlier version of this cell
# assumed) is fragile -- it depends on that sync having already caught up
# with the latest commits, which isn't guaranteed, and doesn't work at all
# in a fresh/upload-only Colab session.
#
# If GitHub is unreachable when you run this (clone fails), you have two
# options instead of waiting: (1) if MyDrive already has an up-to-date
# `src/genewriter` synced into it some other way, just set REPO_DIR below to
# point at that Drive path instead: REPO_DIR = '/content/drive/MyDrive'
# (after the Drive-mount cell below has run); or (2) manually upload the
# `src` folder via Colab's file browser (drag-and-drop works) to wherever,
# and set REPO_DIR to its parent. Either way, nothing past this cell cares
# how `src/genewriter` got here, only that REPO_DIR/src/genewriter exists.
REPO_DIR = '/content/GeneWriter'
!git clone https://github.com/LukeTheGeneWriter/GeneWriter.git {REPO_DIR}

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive

# cupy-cuda12x per this repo's existing GPU setup convention (see
# scripts/colab_stress_test.py) -- Colab's runtime is CUDA 12.x regardless
# of which GPU tier is attached. No `import cupy` in this cell or anywhere
# else at notebook top level -- see baseline_pipeline.py's fork-safety note
# for why that matters (cupy is only ever imported inside an already-forked
# wave-2 child process, never in this notebook's own process).
!pip install --quiet cupy-cuda12x psutil

In [ ]:
import os
import sys

src_dir = os.path.join(REPO_DIR, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from genewriter import baseline_pipeline, standards_io

## Configure

This cell **is** "the list of tests and their save directories," literally -- add or remove a bioinformatic test by editing `TEST_SPECS` (or build your own `list[baseline_pipeline.TestSpec]` instead of using `default_test_specs()`).

In [ ]:
GENE_DIR = 'RefGenes/NHGeneBodySupp'
STANDARDS_DIR = 'Standards'

# Bounds peak resident gene-object memory per chunk -- tune down if you hit
# memory pressure, up if chunk/fork overhead ever becomes a bottleneck (it
# shouldn't: ~26 chunks x 1 shared fork/chunk is negligible next to real compute).
CHUNK_SIZE = 750

# Threads used to load each chunk's gene JSONs concurrently -- the real
# lever for Drive-mounted load time, which is network/FUSE round-trip
# *latency* per file, not local CPU work (Colab's single core doesn't limit
# this the way it would for actual compute -- see gene_io.py's own
# docstring). Tune down if you see Drive API rate-limit errors, up if a
# real run shows headroom.
LOAD_WORKERS = 8

K_VALUES = range(2, 11)  # the real kmer range Kmer_Analysis.ipynb used to compute (k=2..10)
USE_GPU_FOR_KMER = True
USE_GPU_FOR_BASELINES = True  # rare_codon/codon_usage/codon_pair_bias/gc -- separate toggle from kmer's own
ORGANISM = 'human'

TEST_SPECS = baseline_pipeline.default_test_specs(STANDARDS_DIR, K_VALUES, USE_GPU_FOR_KMER, USE_GPU_FOR_BASELINES)
[(s.name, s.wave, s.shard_dir) for s in TEST_SPECS]

## Run

Resumable: if this cell is interrupted and rerun, chunks whose shards already exist for every test are skipped without rereading their gene JSONs.

Expect to see the 5 tests' progress land one after another per chunk (not 4-then-1, or all 5 at once) -- they now share a single forked process and GPU context per chunk (see the intro cell above), so `nvidia-smi`/Colab's resource panel will show one steady GPU user at a time, not several concurrent ones.

In [ ]:
problems = baseline_pipeline.run_pipeline(
    GENE_DIR, STANDARDS_DIR, chunk_size=CHUNK_SIZE, organism=ORGANISM, tests=TEST_SPECS, load_workers=LOAD_WORKERS,
)

if problems:
    print(f"{len(problems)} chunk(s) had at least one test failure -- rerun this cell (resume=True by default) to retry them:")
    print(problems)
else:
    print("All chunks completed with no failures.")

## Finalize

Merges every test's chunk shards into the final `Standards/<Name>Analysis.json` files, then round-trips them through `standards_io.load_standards()` as an end-to-end smoke test that the GA scoring code's exact contract is satisfied.

In [ ]:
written = baseline_pipeline.finalize_all(STANDARDS_DIR, TEST_SPECS, organism=ORGANISM)
print(written)

loaded = standards_io.load_standards(STANDARDS_DIR)
print("Loaded OK:", loaded.rare_codon.totalCodons, "total codons across the corpus")

## (Optional) Cleanup

`Standards/_partial/` (the per-chunk shards) can be deleted once `finalize_all()` has succeeded -- left in place by default for crash-tolerance/inspection. Only run this once you're confident the final `Standards/*.json` files above are correct.

In [ ]:
# import shutil, os
# shutil.rmtree(os.path.join(STANDARDS_DIR, '_partial'))